# 1. Load Data

In [1]:
# Load data
import os
import pandas as pd
import glob
import json
import numpy as np
from tqdm.notebook import tqdm
import torch
import gc
import ipywidgets
from joblib import Parallel, delayed
import geopandas as gpd
from sklearn.neighbors import BallTree, NearestNeighbors


# Confirm the current working directory
print("Current working directory:", os.getcwd())

Current working directory: /home/qusta100/STGNN/02_Notebooks/01_Preprocessing


In [204]:
# Path to the price data folder
folder_path = "/gpfs/scratch/qusta100/STGNN/01_Data/01_Original_Data/01_Prices/"

# Find all CSV files matching the pattern "*-prices.csv"
csv_files = glob.glob(os.path.join(folder_path, "*-prices.csv"))

# Read each file and add it to the list
df_list = []
for file in csv_files:
    df_day = pd.read_csv(file, usecols=[0, 1, 2, 3, 4])
    df_day.columns = ["date", "station_uuid", "diesel", "e5", "e10"]
    df_list.append(df_day)

if not df_list:
    raise ValueError("Keine Dateien gefunden, die auf '*-prices.csv' passen.")

df_prices = pd.concat(df_list, ignore_index=True)

# Parse timestamps and remove timezone
df_prices["date"] = pd.to_datetime(df_prices["date"], utc=True).dt.tz_convert(None)

df_prices = df_prices.sort_values(["station_uuid", "date"])

# Get the overall time range (start & end)
start = df_prices["date"].min().replace(minute=0, second=0)
end = df_prices["date"].max().replace(minute=45, second=0)

# Create 15-minute time intervals
quarter_hours = pd.date_range(start=start, end=end, freq="15min")

In [210]:
# Load all station files from March and April
station_files = glob.glob("/gpfs/scratch/qusta100/Gasoline/01_Data/01_Original_Data/02_Stations/2025-0[3-5]-*-stations.csv")

stations_list = []
for file in station_files:
    df_station = pd.read_csv(file)
    # Extract the date from the filename and convert to datetime
    date_str = os.path.basename(file).split("-stations.csv")[0]
    df_station["day"] = pd.to_datetime(date_str)
    stations_list.append(df_station)

# Combine all station data into a single DataFrame
stations_all = pd.concat(stations_list, ignore_index=True)

In [238]:
# Merge Data Sets
left = df_prices.copy()
right = stations_all.copy()

left["date"] = pd.to_datetime(left["date"], errors="raise")
right["day"] = pd.to_datetime(right["day"], errors="raise")

right = right.rename(columns={"uuid": "station_uuid"})

left  = left.sort_values(["date", "station_uuid"], kind="mergesort").reset_index(drop=True)
right = right.sort_values(["day", "station_uuid"], kind="mergesort").reset_index(drop=True)

# 1) backward
bwd = pd.merge_asof(
    left, right,
    left_on="date", right_on="day",
    by="station_uuid",
    direction="backward",
    allow_exact_matches=True,
    suffixes=("", "_r")
)

# 2) forward
fwd = pd.merge_asof(
    left, right,
    left_on="date", right_on="day",
    by="station_uuid",
    direction="forward",
    allow_exact_matches=True,
    suffixes=("", "_r")
)

# Distanz in Tagen/Timedelta (NaT bleibt NaT)
bwd_dist = bwd["date"] - bwd["day"]
fwd_dist = fwd["day"] - fwd["date"]

# Welche Zeilen sollen forward nehmen?
use_fwd = bwd["day"].isna()


df = bwd.copy()
df.loc[use_fwd, :] = fwd.loc[use_fwd, :]

In [241]:
mask = df["date"] >= "2025-04-01"
df.loc[mask & df["day"].isna(), "station_uuid"].unique()

array([], dtype=object)

In [242]:
df.loc[mask].isna().sum()

date                      0
station_uuid              0
diesel                    0
e5                        0
e10                       0
name                      3
brand                270885
street                  382
house_number         494167
post_code               382
city                   1229
latitude                  0
longitude                 0
first_active              0
openingtimes_json         0
day                       0
dtype: int64

# 2. Sampling

In [243]:
# Clean coordinates
def to_scalar(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return v[0] if len(v) else np.nan
    return v

lon = pd.to_numeric(df["longitude"].map(to_scalar), errors="coerce")
lat = pd.to_numeric(df["latitude"].map(to_scalar),  errors="coerce")

# 2) Vectorize Coordinates
geom = gpd.points_from_xy(lon, lat)  # erwartet WGS84

gdf = gpd.GeoDataFrame(df.copy(), geometry=geom, crs="EPSG:4326")

# 3) Load shape files
nrw = gpd.read_file("/gpfs/scratch/qusta100/STGNN/01_Data/03_Shapefiles/thuringia.geojson").to_crs("EPSG:4326")

# 4) Spatial Join
joined = gpd.sjoin(gdf, nrw[["geometry"]], how="left", predicate="within")
gdf["in_thuringia"] = joined.index_right.notna()

In [244]:
# Initialize in_region: 1 for Thuringia, 0 otherwise
gdf["in_region"] = gdf["in_thuringia"].fillna(False).astype(np.int8)

# Find neighbors within 30 km using BallTree (memory-efficient batch version)
valid = gdf["latitude"].notna() & gdf["longitude"].notna()
th_mask = gdf["in_thuringia"].fillna(False) & valid

if th_mask.any():
    lat_v = gdf.loc[valid, "latitude"].to_numpy(dtype=np.float32)
    lon_v = gdf.loc[valid, "longitude"].to_numpy(dtype=np.float32)
    coords_rad = np.deg2rad(np.column_stack([lat_v, lon_v])).astype(np.float32, copy=False)

    # Store the DataFrame indices of valid rows (to map results back later)
    valid_idx = gdf.index[valid].to_numpy()

    # Query points: only Thuringia stations
    th_lat = gdf.loc[th_mask, "latitude"].to_numpy(dtype=np.float32)
    th_lon = gdf.loc[th_mask, "longitude"].to_numpy(dtype=np.float32)
    th_coords_rad = np.deg2rad(np.column_stack([th_lat, th_lon])).astype(np.float32, copy=False)

    # Build BallTree using Haversine distance (spherical)
    tree = BallTree(coords_rad, metric="haversine")

    # Radius in radians (30 km on Earth’s surface)
    R_EARTH_KM = np.float32(6371.0088)
    radius_rad = np.float32(30.0) / R_EARTH_KM

    # Boolean mask for all valid rows; will be True for stations in the region
    region_mask_valid = np.zeros(coords_rad.shape[0], dtype=bool)

    # Batch query
    BATCH = 5000
    for start in range(0, th_coords_rad.shape[0], BATCH):
        end = start + BATCH
        neigh_lists = tree.query_radius(th_coords_rad[start:end], r=radius_rad)

        # Instead of concatenating large arrays, mark neighbors directly in the mask
        for arr in neigh_lists:
            if arr.size:
                region_mask_valid[arr] = True

    # Ensure all Thuringia points themselves are included
    region_mask_valid[np.searchsorted(valid_idx, gdf.index[th_mask])] = True

    # Map back to global DataFrame indices and set in_region = 1
    gdf.loc[valid_idx[region_mask_valid], "in_region"] = 1

# 6) Create a plain DataFrame without geometry
ndf = pd.DataFrame(gdf).drop(columns="geometry")

In [245]:
# Number of entries in Thuringia
print(len(ndf[ndf["in_thuringia"] == True]))
print(len(ndf[ndf["in_region"] == 1]))

# Filter only entries in the wanted region
ndf = ndf[ndf["in_region"]==1]

823575
1943171


# 3. Process Data

In [248]:
# Result list
resampled_list = []

# List of fuel price columns
price_cols = ["diesel", "e5", "e10"]

# List of metadata columns (everything except date, UUID, and prices)
metadata_cols = [col for col in ndf.columns if col not in ["date", "station_uuid"] + price_cols]

# For each station: resample time series and attach metadata
for station_id, group in ndf.groupby("station_uuid"):
    group = group.set_index("date").sort_index()

    # Resample prices (forward fill) for each fuel type
    reindexed_prices = {
        col: group[col].reindex(quarter_hours, method="ffill")
        for col in price_cols
    }

    # DataFrame with timestamps and UUID
    result = pd.DataFrame({
        "date": quarter_hours,
        "station_uuid": station_id,
        **{col: reindexed_prices[col].values for col in price_cols}
    })

    # Metadata from the first row (if available)
    meta = group[metadata_cols].iloc[0] if not group.empty else pd.Series(index=metadata_cols)

    # Attach metadata to each row
    for col in metadata_cols:
        result[col] = meta[col]

    resampled_list.append(result)

# Merge all resampled data
resampled_df = pd.concat(resampled_list, ignore_index=True)

# Restrict to April
resampled_df = resampled_df[resampled_df["date"] >= pd.Timestamp("2025-04-01 00:00:00")]

In [251]:
# Remove rows with all prices missing
df_clean = df_clean[df_clean[["diesel", "e5", "e10"]].notna().any(axis=1)].copy()

In [252]:
df_clean["date"] = pd.to_datetime(df_clean["date"])

# 1) Last observation date per station
last_seen = (
    df_clean.groupby("station_uuid", as_index=False)["date"]
    .max()
    .rename(columns={"date": "last_seen"})
)

# 2) Create a new DataFrame
df_filtered = df_clean.merge(last_seen, on="station_uuid", how="left")
df_filtered["last_seen"] = pd.to_datetime(df_filtered["last_seen"])

# 3) Moving activity and price filter
cutoff = pd.Timedelta(days=15)
mask_active = (df_filtered["date"] - df_filtered["last_seen"]) <= cutoff
mask_price  = df_filtered["diesel"] > 0

df_filtered = df_filtered[mask_active & mask_price].copy()

print("Obs after filter:", len(df_filtered))

Obs after filter: 2963412


In [265]:
# Replace 0 with NaN for all price columns
price_columns = ["diesel", "e5", "e10"]

for col in price_columns:
    if col in df_filtered.columns:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors="coerce")
        df_filtered.loc[df_filtered[col] <= 0, col] = np.nan

In [254]:
# Handling opening times
HOLIDAY_BIT = 1 << 7  # 128

def is_open(opening_json, timestamp, holiday: bool):
    try:
        if opening_json.strip() == "{}":
            return True

        data = json.loads(opening_json)

        # 1) Overrides haben Vorrang
        for ov in data.get("overrides", []):
            start = pd.to_datetime(ov["startp"])
            end = pd.to_datetime(ov["endp"])
            if start <= timestamp < end:
                return not ov.get("is_close", False)

        # 2) Regular openingTimes
        weekday = timestamp.weekday()      # 0=Mo ... 6=So
        mask = 1 << weekday                # 1,2,4,8,16,32,64
        if holiday:
            mask |= HOLIDAY_BIT             # +128

        time_str = timestamp.strftime("%H:%M")

        for block in data.get("openingTimes", []):
            if block.get("applicable_days", 0) & mask:
                for period in block.get("periods", []):
                    if period["startp"] <= time_str < period["endp"]:
                        return True

        return False
    except:
        return True
    
    
# Create a copy
df_clean_opening = df_filtered.copy()

# Holiday dummy (1 if date is 2025-04-18 or 2025-04-21, otherwise 0)
holidays = {pd.Timestamp('2025-04-18'), pd.Timestamp('2025-04-21')}

# .normalize() resets the time to midnight so you compare only dates
df_clean_opening['holiday'] = df_clean_opening['date'].dt.normalize().isin(holidays).astype(int)


# Price columns to set to NaN if station is closed
price_columns = [col for col in ["diesel", "e5", "e10"] if col in df_clean_opening.columns]


df_clean_opening["is_open"] = df_clean_opening.apply(
    lambda row: is_open(
        row["openingtimes_json"],
        row["date"],
        row["holiday"]
    ),
    axis=1
)

# Set all prices to NaN if closed
for col in price_columns:
    df_clean_opening.loc[~df_clean_opening["is_open"], col] = pd.NA

In [276]:
# Show data to validate function for opening times
df_clean_opening[df_clean_opening['is_open'] == False].head(5)

,date,station_uuid,diesel,e5,e10,uuid,name,brand,street,house_number,...,city,latitude,longitude,first_active,openingtimes_json,in_thuringia,in_region,last_seen,holiday,is_open
28720,2025-04-01 00:00:00,00060860-0005-4444-8888-acdc00000005,NaN,NaN,NaN,00060860-0005-4444-8888-acdc00000005,Wilfried Lehmann Mineralölhandel,Tank & Waschcenter Auerbach,Klingenthalerstr. 111,NaN,...,Auerbach,50.496922,12.42044,2014-08-22 00:00:01+02,"{""openingTimes"":[{""applicable_days"":31,""periods"":[{""startp"":""06:00"",""endp"":""20:00""}]},{""applicable_days"":32,""periods"":[{""startp"":""07:00"",""endp"":""20:00""}]},{""applicable_days"":64,""periods"":[{""startp"":""08:00"",""endp"":""20:00""}]}]}",False,1,2025-04-30 21:45:00,0,False
28721,2025-04-01 00:15:00,00060860-0005-4444-8888-acdc00000005,NaN,NaN,NaN,00060860-0005-4444-8888-acdc00000005,Wilfried Lehmann Mineralölhandel,Tank & Waschcenter Auerbach,Klingenthalerstr. 111,NaN,...,Auerbach,50.496922,12.42044,2014-08-22 00:00:01+02,"{""openingTimes"":[{""applicable_days"":31,""periods"":[{""startp"":""06:00"",""endp"":""20:00""}]},{""applicable_days"":32,""periods"":[{""startp"":""07:00"",""endp"":""20:00""}]},{""applicable_days"":64,""periods"":[{""startp"":""08:00"",""endp"":""20:00""}]}]}",False,1,2025-04-30 21:45:00,0,False
28722,2025-04-01 00:30:00,00060860-0005-4444-8888-acdc00000005,NaN,NaN,NaN,00060860-0005-4444-8888-acdc00000005,Wilfried Lehmann Mineralölhandel,Tank & Waschcenter Auerbach,Klingenthalerstr. 111,NaN,...,Auerbach,50.496922,12.42044,2014-08-22 00:00:01+02,"{""openingTimes"":[{""applicable_days"":31,""periods"":[{""startp"":""06:00"",""endp"":""20:00""}]},{""applicable_days"":32,""periods"":[{""startp"":""07:00"",""endp"":""20:00""}]},{""applicable_days"":64,""periods"":[{""startp"":""08:00"",""endp"":""20:00""}]}]}",False,1,2025-04-30 21:45:00,0,False
28723,2025-04-01 00:45:00,00060860-0005-4444-8888-acdc00000005,NaN,NaN,NaN,00060860-0005-4444-8888-acdc00000005,Wilfried Lehmann Mineralölhandel,Tank & Waschcenter Auerbach,Klingenthalerstr. 111,NaN,...,Auerbach,50.496922,12.42044,2014-08-22 00:00:01+02,"{""openingTimes"":[{""applicable_days"":31,""periods"":[{""startp"":""06:00"",""endp"":""20:00""}]},{""applicable_days"":32,""periods"":[{""startp"":""07:00"",""endp"":""20:00""}]},{""applicable_days"":64,""periods"":[{""startp"":""08:00"",""endp"":""20:00""}]}]}",False,1,2025-04-30 21:45:00,0,False
28724,2025-04-01 01:00:00,00060860-0005-4444-8888-acdc00000005,NaN,NaN,NaN,00060860-0005-4444-8888-acdc00000005,Wilfried Lehmann Mineralölhandel,Tank & Waschcenter Auerbach,Klingenthalerstr. 111,NaN,...,Auerbach,50.496922,12.42044,2014-08-22 00:00:01+02,"{""openingTimes"":[{""applicable_days"":31,""periods"":[{""startp"":""06:00"",""endp"":""20:00""}]},{""applicable_days"":32,""periods"":[{""startp"":""07:00"",""endp"":""20:00""}]},{""applicable_days"":64,""periods"":[{""startp"":""08:00"",""endp"":""20:00""}]}]}",False,1,2025-04-30 21:45:00,0,False


# 4. Adding more features

In [293]:
# Time/Day Features
df = df_clean_opening.copy()

# Assuming your DataFrame is named df and the timestamp column is 'date'
s = df['date'].astype(str) \
              .str.replace('\u00a0', ' ', regex=False) \
              .str.strip() \
              .str.replace(r'\s+', ' ', regex=True)
parsed_full  = pd.to_datetime(s, format='%Y-%m-%d %H:%M:%S', errors='coerce')
parsed_date  = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
df['date']   = parsed_full.fillna(parsed_date)

# 1) Weekday (Monday–Sunday)
df["weekday"] = df["date"].dt.weekday.astype(int)

# 2) Time formatted as "hh:mm"
minutes = df["date"].dt.hour * 60 + df["date"].dt.minute
df["time_sin"] = np.sin(2 * np.pi * minutes / 1440)
df["time_cos"] = np.cos(2 * np.pi * minutes / 1440)

# 3) One Hot Encodings
df["weekday"] = df["date"].dt.weekday
df = pd.get_dummies(df, columns=["weekday"], prefix="wd")

slot = (df["date"].dt.hour * 60 + df["date"].dt.minute) // 15
time_oh = pd.get_dummies(slot, prefix="t")
df = pd.concat([df, time_oh], axis=1)


# Example output
print(df[['date', 'holiday', 'time_sin', 'time_cos']].head())

                 date  holiday  time_sin  time_cos
0 2025-04-01 00:00:00        0  0.000000  1.000000
1 2025-04-01 00:15:00        0  0.065403  0.997859
2 2025-04-01 00:30:00        0  0.130526  0.991445
3 2025-04-01 00:45:00        0  0.195090  0.980785
4 2025-04-01 01:00:00        0  0.258819  0.965926


In [294]:
def clean_brand(df, brand_col="brand", new_col="brand_cat"):
    # create new variable
    df[new_col] = df[brand_col].copy()

    # 1) Direct renaming
    df.loc[df[new_col] == "AGIP ENI", new_col] = "Agip"
    df.loc[df[new_col] == "AVIA XPress", new_col] = "AVIA"
    df.loc[df[new_col] == "ORLEN Express", new_col] = "OLEN"
    df.loc[df[new_col] == "SB-Markttankstelle", new_col] = "SB"
    df.loc[df[new_col] == "TotalEnergies Truckstop", new_col] = "TotalEnergies"

    # 2) Starts-with rules (case-insensitive where useful)
    mask_bft = df[new_col].str.lower().str.startswith("bft", na=False)
    df.loc[mask_bft, new_col] = "bft"

    mask_hmh = df[new_col].str.startswith("HMH MineralÃ¶l Service", na=False)
    df.loc[mask_hmh, new_col] = "HMH"

    mask_raiff = df[new_col].str.startswith("Raiffeisen", na=False)
    df.loc[mask_raiff, new_col] = "Raiffeisen"

    mask_frei = df[new_col].str.lower().str.startswith("frei", na=False)
    df.loc[mask_frei, new_col] = np.nan

    # 3) Remove all "supermarket" cases
    mask_supermarkt_tank = df[new_col].str.contains("Supermarkt-Tankstelle", na=False)
    mask_supermarkt = df[new_col].str.contains("Supermarkt", na=False)

    df.loc[mask_supermarkt_tank | mask_supermarkt, new_col] = np.nan

    # 4) Remove all brands that appear only once
    counts = df[new_col].value_counts(dropna=True)
    singletons = counts[counts == 1].index
    df.loc[df[new_col].isin(singletons), new_col] = np.nan
    
    df[new_col] = df[new_col].astype("category").cat.codes.astype(float)

    return df

df = clean_brand(df, brand_col="brand", new_col="brand_cat")

In [ ]:
# Import raw oil prices via API
# Because of a missing network connection here, this step was run on a different server

import yfinance as yf
import pandas as pd

# 1. Retrieve hourly Brent data from April 1 to April 30, 2025
ticker = yf.Ticker("BZ=F")
df = ticker.history(start="2025-04-01", end="2025-05-01", interval="60m")
df = df.reset_index()
df['Datetime'] = df['Datetime'].dt.tz_localize(None)

# 2. Generate a complete 15-minute timestamp grid for April 2025
all_times = pd.date_range(
    start="2025-04-01 00:00:00",
    end="2025-04-30 23:45:00",
    freq="15min"
)
full_df = pd.DataFrame({'Datetime': all_times})

# 3. Select only the 'Open' price and rename it to 'Brent_Price'
df_short = df[['Datetime', 'Open']].rename(columns={'Open': 'Brent_Price'})

# 4. Merge the full 15-minute grid with the hourly prices,
#    filling each 15-min slot with the most recent (backward fill) hourly price
merged = pd.merge_asof(
    full_df.sort_values('Datetime'),
    df_short.sort_values('Datetime'),
    on='Datetime',
    direction='backward'
)

# 5. Save the result to a CSV file
merged.to_csv("/gpfs/scratch/qusta100/STGNN/01_Data/02_Processed_Data/02_Analysis/brent_april_2025_15min_filled.csv", index=False)

# Optional: display the first 20 rows and total count (should be 2880 timestamps)
print(merged.head(20))
print("Total number of timestamps:", len(merged))

In [295]:
# Load data (restore from saved file)
brent = pd.read_csv("/gpfs/scratch/qusta100/STGNN/01_Data/02_Processed_Data/02_Analysis/brent_april_2025_15min_filled.csv")
brent = brent.rename(columns={"Datetime": "date"})
brent['date'] = pd.to_datetime(brent['date'])

In [296]:
# Merge with main data set (df_main)
final_df = pd.merge(df, brent, on='date', how='left')

# Show data set
final_df.head()

,date,station_uuid,diesel,e5,e10,uuid,name,brand,street,house_number,...,t_88,t_89,t_90,t_91,t_92,t_93,t_94,t_95,brand_cat,Brent_Price
0,2025-04-01 00:00:00,00060065-7890-4444-8888-acdc00000004,1.559,1.709,1.649,00060065-7890-4444-8888-acdc00000004,Georg Ultsch GmbH,Tankstelle Lichtenfels,Robert-Koch-Str.,18,...,False,False,False,False,False,False,False,False,74.0,74.959999
1,2025-04-01 00:15:00,00060065-7890-4444-8888-acdc00000004,1.559,1.709,1.649,00060065-7890-4444-8888-acdc00000004,Georg Ultsch GmbH,Tankstelle Lichtenfels,Robert-Koch-Str.,18,...,False,False,False,False,False,False,False,False,74.0,74.959999
2,2025-04-01 00:30:00,00060065-7890-4444-8888-acdc00000004,1.559,1.709,1.649,00060065-7890-4444-8888-acdc00000004,Georg Ultsch GmbH,Tankstelle Lichtenfels,Robert-Koch-Str.,18,...,False,False,False,False,False,False,False,False,74.0,74.959999
3,2025-04-01 00:45:00,00060065-7890-4444-8888-acdc00000004,1.559,1.709,1.649,00060065-7890-4444-8888-acdc00000004,Georg Ultsch GmbH,Tankstelle Lichtenfels,Robert-Koch-Str.,18,...,False,False,False,False,False,False,False,False,74.0,74.959999
4,2025-04-01 01:00:00,00060065-7890-4444-8888-acdc00000004,1.559,1.709,1.649,00060065-7890-4444-8888-acdc00000004,Georg Ultsch GmbH,Tankstelle Lichtenfels,Robert-Koch-Str.,18,...,False,False,False,False,False,False,False,False,74.0,74.959999


# 5. Save the files

In [297]:
# Save the final dataframe to CSV
final_df.to_csv("/gpfs/scratch/qusta100/STGNN/01_Data/02_Processed_Data/02_Analysis/final_df.csv", index=False)

In [282]:
# Verification and exclusion of highway stations using the stations dataset
autobahn_pattern = (
    r"\b(?:"
    r"autobahn(?:\s*tankstelle)?"
    r"|autohof"
    r"|rasthof"
    r"|rast[-\s]?stätte"
    r"|raststaette"
    r"|tank\s*(?:und|&)\s*rast"
    r"|service\s*area"
    r"|an\s+der\s+a[0-9]{1,2}"
    r"|a\s*[0-9]{1,2}"
    r"|ausfahrt"
    r"|abfahrt"
    r"|bundesautobahn"
    r")\b"
)

# 2. Identify highway stations by name or street
is_highway = (
    stations_latest["name"].str.contains(autobahn_pattern, case=False, na=False) |
    stations_latest["street"].str.contains(autobahn_pattern, case=False, na=False)
)

# 3. Get UUIDs of identified highway stations
highway_ids = stations_latest[is_highway]["uuid"].unique()

# 4. Exclude these stations from df_clean_opening
df_excl_highway = final_df[
    ~final_df["station_uuid"].isin(highway_ids)
].copy()

# 5. Output
print("Number of detected highway stations:", len(highway_ids))
print("Rows before exclusion:", len(final_df))
print("Rows after exclusion:", len(df_excl_highway))

Number of detected highway stations: 452
Rows before exclusion: 2963412
Rows after exclusion: 2868636


In [ ]:
# Save dataframe without highways to CSV
final_df.to_csv("/gpfs/scratch/qusta100/STGNN/01_Data/02_Processed_Data/02_Analysis/final_no_highways.csv", index=False)